# 06 — Benchmark Summary

**FairLens AI / NyayaLens — Person 3 (ML Layer)**

This notebook brings everything together: runs the full pipeline from
the modular `ml/src/` modules and produces a final summary.

### What we do here
1. Import our modular `ml/src/` functions
2. Run the full pipeline end-to-end
3. Display a complete summary table
4. Produce final visualizations
5. Save the final report JSON

In [ ]:
import sys
import os
import json

# Add project root to path so we can import ml.src modules
project_root = os.path.abspath(os.path.join("..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"Project root: {project_root}")
print("Libraries loaded.")

## Step 1 — Import our modular pipeline functions

These are the same functions Person 2's backend will import.

In [ ]:
from ml.src.data_loader import load_adult_dataset
from ml.src.preprocessing import preprocess
from ml.src.train import train_model, evaluate_model
from ml.src.evaluate import detect_bias
from ml.src.mitigation import apply_mitigation

print("All ml.src modules imported successfully.")

## Step 2 — Run the full pipeline

In [ ]:
# --- Load ---
raw_df = load_adult_dataset()
print(f"Loaded: {raw_df.shape}")

In [ ]:
# --- Preprocess ---
# Use target_col='class' because OpenML names the target 'class'
prep_result = preprocess(raw_df, target_col="class", sensitive_cols=["sex"])
print(f"Preprocess status: {prep_result['status']}")
print(f"Train shape: {prep_result['features_train'].shape}")
print(f"Test shape:  {prep_result['features_test'].shape}")

In [ ]:
# --- Train ---
model = train_model(
    prep_result["features_train"],
    prep_result["income_labels_train"]
)
print(f"Model trained. Iterations: {model.n_iter_[0]}")

In [ ]:
# --- Evaluate baseline ---
baseline_preds = model.predict(prep_result["features_test"])
baseline_metrics = evaluate_model(baseline_preds, prep_result["income_labels_test"])
print("Baseline metrics:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# --- Detect bias ---
bias_result = detect_bias(
    baseline_preds,
    prep_result["income_labels_test"],
    prep_result["sensitive_test"],
)
print("Bias detection:")
print(f"  Status: {bias_result['status']}")
print(f"  By group: {json.dumps(bias_result['metrics']['by_group'], indent=4)}")
print(f"  Differences: {bias_result['metrics']['differences']}")
print(f"  Findings: {len(bias_result['findings'])} issue(s)")
for finding in bias_result['findings']:
    print(f"    [{finding['severity']}] {finding['description']}")

In [ ]:
# --- Mitigate ---
mitigation_result = apply_mitigation(
    model,
    prep_result["features_train"],
    prep_result["income_labels_train"],
    prep_result["sensitive_train"],
    prep_result["features_test"],
    prep_result["income_labels_test"],
    prep_result["sensitive_test"],
)
print(f"Mitigation status: {mitigation_result['status']}")
print(f"Improvement: {json.dumps(mitigation_result['improvement'], indent=2)}")

## Step 3 — Complete summary table

In [ ]:
before = mitigation_result["before"]
after = mitigation_result["after"]

print(f"{'Metric':<30} {'Before':>10} {'After':>10} {'Change':>10}")
print("=" * 62)

for metric in ["accuracy", "precision", "recall", "f1"]:
    bv = before["model_metrics"][metric]
    av = after["model_metrics"][metric]
    print(f"{metric.capitalize():<30} {bv:>10.4f} {av:>10.4f} {av - bv:>+10.4f}")

print()
before_groups = before["bias_detection"]["metrics"]["by_group"]
after_groups = after["bias_detection"]["metrics"]["by_group"]

for group in before_groups:
    br = before_groups[group]["recall"]
    ar = after_groups[group]["recall"]
    print(f"{'Recall (' + group + ')':<30} {br:>10.4f} {ar:>10.4f} {ar - br:>+10.4f}")

bd = before["bias_detection"]["metrics"]["differences"]["recall_difference"]
ad = after["bias_detection"]["metrics"]["differences"]["recall_difference"]
print(f"{'Recall Difference':<30} {bd:>10.4f} {ad:>10.4f} {ad - bd:>+10.4f}")

## Step 4 — Final visualizations

In [ ]:
groups = list(before_groups.keys())
x = np.arange(len(groups))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Recall by group ---
br_vals = [before_groups[g]["recall"] for g in groups]
ar_vals = [after_groups[g]["recall"] for g in groups]
axes[0].bar(x - width/2, br_vals, width, label='Before', color='#F44336', alpha=0.8)
axes[0].bar(x + width/2, ar_vals, width, label='After', color='#4CAF50', alpha=0.8)
axes[0].set_title('Recall by Group')
axes[0].set_ylabel('Recall')
axes[0].set_xticks(x)
axes[0].set_xticklabels(groups)
axes[0].set_ylim(0, 1)
axes[0].legend()

# --- Accuracy by group ---
ba_vals = [before_groups[g]["accuracy"] for g in groups]
aa_vals = [after_groups[g]["accuracy"] for g in groups]
axes[1].bar(x - width/2, ba_vals, width, label='Before', color='#F44336', alpha=0.8)
axes[1].bar(x + width/2, aa_vals, width, label='After', color='#4CAF50', alpha=0.8)
axes[1].set_title('Accuracy by Group')
axes[1].set_ylabel('Accuracy')
axes[1].set_xticks(x)
axes[1].set_xticklabels(groups)
axes[1].set_ylim(0.5, 1)
axes[1].legend()

# --- Disparity reduction ---
axes[2].bar(['Before', 'After'], [bd, ad], color=['#F44336', '#4CAF50'], alpha=0.8)
axes[2].set_title('Recall Disparity')
axes[2].set_ylabel('Max Recall Gap')
axes[2].axhline(y=0.10, color='orange', linestyle='--', label='HIGH (10%)')
axes[2].axhline(y=0.05, color='gray', linestyle='--', label='MEDIUM (5%)')
axes[2].legend()

plt.suptitle('FairLens AI — Adult Dataset Audit Summary', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Step 5 — Save final report

In [ ]:
final_report = {
    "status": "success",
    "dataset": "UCI Adult (Census Income)",
    "sensitive_columns": ["sex"],
    "target_column": "class",
}
final_report.update(mitigation_result)

output_path = os.path.join("..", "outputs", "reports", "adult_benchmark.json")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    json.dump(final_report, f, indent=2)

print(f"Final report saved to: {output_path}")
print()
print(json.dumps(final_report, indent=2))

## Summary

This notebook demonstrates the full FairLens AI pipeline using our
modular `ml/src/` code:

1. **data_loader.py** — loads the Adult dataset
2. **preprocessing.py** — cleans, encodes, scales, splits
3. **train.py** — trains Logistic Regression and evaluates
4. **evaluate.py** — detects bias using Fairlearn MetricFrame
5. **mitigation.py** — applies ThresholdOptimizer and compares

The JSON output is ready for Person 2's FastAPI backend to return
from `POST /audits`.